# Mapperatorinator – local beatmap generation

This notebook is a local version of the [Mapperatorinator](https://github.com/OliBomby/Mapperatorinator) Colab demo by OliBomby. The model generates hit objects, hitsounds, timing, kiai times, and slider velocity for all 4 gamemodes. The environment is managed with uv and runs on an NVIDIA GPU (RTX 3090 and 4090).

The Hungarian notebook is `mapperatorinator_inference.ipynb`.

### Use this tool responsibly

Always disclose the use of AI in your beatmaps.

### How to run

* Accept the rules with the checkbox at the top of the form.
* Run the cells from top to bottom.
* Install, then start. Double-click `scripts\install.bat`, then `scripts\start-en.bat`.

```powershell
.\scripts\install.ps1
.\scripts\start-en.ps1
```

* **Audio:** `.mp3`, `.ogg`, `.wav`, `.flac`, or `.m4a`.
* **Reference beatmap,** when you need one: a `.osu` file. In stable, use File → Open Song Folder. In lazer, use File → Edit Externally. Required for in-context options and for remapping a section. If the audio sits next to the `.osu`, the audio path can stay empty.
* Configure the style, then press **Generate**. The first run downloads the model from Hugging Face (several GB). After that, a typical song takes a few minutes.
* Enter **-1** to leave a value unknown. V32 is the best model. V30 does not support other gamemodes, year, descriptors, or in-context options.

### Tasks

* **New beatmap:** timing, kiai, hit objects, and slider velocity. Audio is enough. Exported as `.osz`.
* **Timing only:** slower, more accurate super timing. Use this for variable BPM.
* **Guest difficulty:** follow another difficulty's rhythm without copying its patterns.
* **Hitsounds:** keep the reference hit objects and add hitsounds.
* **Section:** regenerate a time range and keep everything before it.
* **Custom:** every control is editable.

Settings are saved to `beallitasok.json`.


In [1]:
from pathlib import Path
import os
import subprocess
import sys

def projekt_mappa() -> Path:
    jelolt = Path.cwd().resolve()
    for mappa in (jelolt, *jelolt.parents):
        if (mappa / "pyproject.toml").exists() and any((mappa / nev).exists() for nev in ("mapperatorinator_inference.ipynb", "mapperatorinator_inference_en.ipynb")):
            return mappa
    return Path(r"C:\Users\Zsolt\Documents\pythons\osu_ai_mapper")


PROJEKT = projekt_mappa()
REPO = PROJEKT / "Mapperatorinator"
FFMPEG_BIN = PROJEKT / "eszkozok" / "ffmpeg" / "bin"
KIMENET = PROJEKT / "kimenet"
BEALLITASOK = PROJEKT / "beallitasok.json"
FFMPEG_URL = "https://github.com/GyanD/codexffmpeg/releases/download/7.1.1/ffmpeg-7.1.1-full_build-shared.zip"

os.environ.setdefault("WANDB_MODE", "disabled")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")


def parancs(argumentumok):
    return subprocess.run(argumentumok, check=True, text=True, capture_output=True)


def biztosit_repot():
    if (REPO / "inference.py").exists():
        return
    print("Downloading the Mapperatorinator submodule...")
    parancs(["git", "-C", str(PROJEKT), "submodule", "update", "--init", "--depth", "1"])


def biztosit_ffmpeg():
    if (FFMPEG_BIN / "ffmpeg.exe").exists() and (FFMPEG_BIN / "ffprobe.exe").exists():
        return
    import io
    import urllib.request
    import zipfile

    print("Downloading the FFmpeg 7.1 shared build for audio...")
    FFMPEG_BIN.parent.parent.mkdir(parents=True, exist_ok=True)
    with urllib.request.urlopen(FFMPEG_URL) as valasz:
        csomag = zipfile.ZipFile(io.BytesIO(valasz.read()))
    cel = PROJEKT / "eszkozok"
    csomag.extractall(cel)
    kibontott = next(cel.glob("ffmpeg-*"))
    if FFMPEG_BIN.parent.exists():
        raise RuntimeError(f"The FFmpeg folder exists but is incomplete: {FFMPEG_BIN.parent}")
    kibontott.rename(cel / "ffmpeg")


def beallit_utvonalakat():
    if str(FFMPEG_BIN) not in os.environ.get("PATH", ""):
        os.environ["PATH"] = str(FFMPEG_BIN) + os.pathsep + os.environ.get("PATH", "")
    if hasattr(os, "add_dll_directory"):
        os.add_dll_directory(str(FFMPEG_BIN))
    if str(REPO) not in sys.path:
        sys.path.insert(0, str(REPO))


biztosit_repot()
biztosit_ffmpeg()
KIMENET.mkdir(exist_ok=True)
beallit_utvonalakat()

if sys.maxsize <= 2**32:
    raise RuntimeError(
        "32-bit Python is running. CUDA PyTorch needs the 64-bit interpreter.\n"
        "Start: .\\scripts\\start-en.ps1"
    )

venv = (PROJEKT / ".venv").resolve()
if Path(sys.prefix).resolve() != venv:
    print("WARNING: this is not the project uv environment, so packages may be missing.")
    print("From PowerShell, in the project folder:")
    print("  .\\scripts\\start.ps1")
    print(f"Or pick this kernel: {venv / 'Scripts' / 'python.exe'}")

import warnings

with warnings.catch_warnings():
    warnings.simplefilter("ignore", SyntaxWarning)
    from pydub import AudioSegment

AudioSegment.converter = str(FFMPEG_BIN / "ffmpeg.exe")
AudioSegment.ffprobe = str(FFMPEG_BIN / "ffprobe.exe")

import torch

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nincs CUDA"
print(f"Python: {sys.version.split()[0]}  ({'64-bit' if sys.maxsize > 2**32 else '32-bit'})")
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {gpu}")
if not torch.cuda.is_available():
    print("Without CUDA, generation is very slow or will not start. Check the driver.")

import inference  # noqa: F401  a ConfigStore a compose előtt kell
from osuT5.osuT5.event import ContextType

verzio = parancs(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"]).stdout.strip()
ffmpeg_verzio = parancs([str(FFMPEG_BIN / "ffmpeg.exe"), "-version"]).stdout.splitlines()[0]
print(f"Mapperatorinator: {verzio}")
print(f"FFmpeg: {ffmpeg_verzio}")
print(f"Kimeneti mappa: {KIMENET}")
print("Environment ready. Run the next cell and fill in the form.")

KORNYEZET_KESZ = True
OSU_SONGS = Path.home() / "AppData" / "Local" / "osu!" / "Songs"


Python: 3.12.14  (64-bit)
PyTorch: 2.10.0+cu128
GPU: NVIDIA GeForce RTX 3090
Mapperatorinator: 0e2b0e3
FFmpeg: ffmpeg version 7.1.1-full_build-www.gyan.dev Copyright (c) 2000-2025 the FFmpeg developers
Kimeneti mappa: C:\Users\Zsolt\Documents\pythons\osu_ai_mapper\kimenet
Environment ready. Run the next cell and fill in the form.


In [2]:
import json
import os
import sys
import time
import traceback
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

if not globals().get("KORNYEZET_KESZ"):
    raise RuntimeError("Run the environment cell above first.")

from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from osuT5.osuT5.event import ContextType

CIMKE_FAJL = REPO / "datasets" / "tags_2026.json"
NYERS_CIMKEK = json.loads(CIMKE_FAJL.read_text(encoding="utf-8"))["tags"]
ISMERT = {}
for tag in NYERS_CIMKEK:
    ISMERT.setdefault(tag["name"], tag)

MAGYAR = {
    "style/messy": "rendezetlen, zsúfolt stílus",
    "expression/simple": "egyszerű",
    "expression/chaotic": "kaotikus",
    "style/geometric": "geometrikus",
    "style/freeform": "szabad formájú",
    "style/grid snap": "rácsra igazított",
    "style/hexgrid": "hatszögrács",
    "style/symmetrical": "szimmetrikus",
    "gimmick/playfield constraint": "a pálya csak egy része használt",
    "expression/old-style revival": "régi mappingstílus",
    "expression/repetition": "ismétlődő minták",
    "expression/progression": "fokozatos nehezedés",
    "expression/high contrast": "nagy kontraszt a részek között",
    "expression/improvisation": "improvizáció",
    "skillset/tech": "tech",
    "tech/slider tech": "slider tech",
    "tech/finger control": "finger control",
    "sliders/complex sv": "bonyolult slider velocity",
    "skillset/jumps": "ugrások",
    "jumps/sharp": "éles szögű ugrások",
    "jumps/wide": "széles szögű ugrások",
    "jumps/linear": "egyenes vonalú ugrások",
    "tech/aim control": "aim control",
    "streams/flow aim": "flow aim",
    "streams/bursts": "burstök",
    "skillset/streams": "streamek",
    "streams/spaced streams": "ritkított streamek",
    "streams/stamina": "hosszú, fárasztó streamek",
    "streams/cutstreams": "kihagyásos streamek",
    "skillset/reading": "olvasás",
    "reading/visually dense": "vizuálisan sűrű",
    "reading/overlaps": "átfedések",
    "skillset/precision": "precíziós aim",
    "skillset/alt": "alternating",
    "meta/collab": "több mapper",
    "meta/multi-song": "több dal egy mapban",
    "meta/variable timing": "változó timing",
    "meta/time signatures": "szokatlan ütemmutató",
    "style/clean": "tiszta, rendezett",
    "sliders/complex slidershapes": "bonyolult sliderformák",
    "gimmick/aspire": "aspire",
    "style/distance snap": "distance snap",
    "context/mapping contest": "mapping verseny",
    "context/tournament custom": "tournament custom",
    "tech/complex snap": "szokatlan snapek",
    "expression/iNiS-style": "iNiS stílus",
    "gimmick/2B": "egyszerre több objektum (2B)",
    "expression/playfield usage": "tudatos pályahasználat",
    "gimmick/tag": "tag multi",
    "expression/difficulty spike": "hirtelen nehézségugrás",
    "style/avant-garde": "avantgárd",
    "additions/keysounds": "keysound dallam",
    "gimmick/storyboard": "storyboard változtat a játékon",
    "sliders/low sv": "alacsony SV",
    "sliders/high sv": "magas SV",
    "meta/mega collab": "mega collab",
    "gimmick/slider only": "csak slider",
    "gimmick/circle only": "csak kör",
    "additions/custom skin": "egyedi skin",
    "gimmick/video": "videókövetés",
    "reading/perfect stacks": "tökéletes stackek",
    "gimmick/ninja spinners": "nagyon rövid spinnerek",
    "meta/accelerating bpm": "gyorsuló BPM",
    "context/custom song": "a maphez írt dal",
    "gimmick/mirrored": "tükrözött minták",
    "style/vocal": "vokálkövetés",
    "style/tnt": "Taiko no Tatsujin stílus",
    "streams/speed": "nagyon gyors tappolás",
    "style/double bpm": "dupla BPM",
    "style/finisher-heavy": "sok finisher",
    "style/mono-heavy": "sok mono",
    "gimmick/reversed": "fordított minták",
    "gimmick/yellow notes": "nagyon rövid sárga noték",
    "gimmick/barlines": "barline-ok",
    "gimmick/memory": "memorizálós",
    "style/taikosu": "osu! és taiko együtt",
    "meta/swing": "swing ritmus",
    "style/convert": "convert stílus",
    "additions/hitsounds": "egyedi hitsoundok",
    "tech/flow": "flow",
    "tech/antiflow": "antiflow",
    "tech/jump": "ugrás / dash",
    "gimmick/no hyperdashes": "hyperdash nélkül",
    "gimmick/long sliders": "hosszú sliderek",
    "gimmick/spinner-heavy": "sok spinner",
    "tech/wiggles": "gyors irányváltások",
    "tech/hyperwalks": "hyperwalk",
    "gimmick/dodge the beat": "az objektumokat kerülni kell",
    "style/jumpstream": "jumpstream",
    "style/handstream": "handstream",
    "style/quadstream": "quadstream",
    "style/chordjack": "chordjack",
    "style/longjack": "longjack",
    "skillset/speedjack": "speedjack",
    "skillset/wristjack": "wristjack",
    "style/chordstream": "chordstream",
    "gimmick/delay": "delay",
    "style/mixed rice": "vegyes rice",
    "style/LN coordination": "LN koordináció",
    "style/LN release": "LN elengedés",
    "gimmick/LN inverse": "fordított LN",
    "style/LN density": "sűrű LN",
    "style/LN mixed ": "vegyes LN",
    "style/generic hybrid": "egyszerű rice + LN",
    "tech/technical hybrid": "technikai rice + LN",
    "style/tiebreaker": "tiebreaker",
    "style/o2jam": "o2jam",
    "style/N+1": "N+1",
    "style/dump": "dump",
    "jumps/squares": "négyzetes ugrások",
    "jumps/triangles": "háromszög ugrások",
    "jumps/cross-screen": "keresztképernyős ugrások",
    "jumps/stamina": "hosszú ugrássorok",
    "skillset/gimmick": "gimmick",
    "jumps/stars": "csillag alakú ugrások",
    "jumps/back and forth": "oda-vissza ugrások",
    "jumps/freeform": "szabad ugrások",
    "streams/doubles": "dupla hangok",
    "streams/quads": "négyes hangok",
    "expression/inspo": "inspiráció egy másik mapből",
    "expression/conceptual": "konceptuális",
    "additions/combo colours": "combo színek",
}

FELADATOK = {
    "uj": {
        "leiras": "A new beatmap is generated from the song: timing, kiai, hit objects, and slider velocity. Audio is enough.",
        "kontextus": "[NONE]",
        "kimenet": "[TIMING,MAP,SV]",
        "hozzaad": False,
        "osz": True,
        "super_timing": False,
        "hitsounded": None,
        "referencia": False,
        "szakasz": False,
    },
    "timing": {
        "leiras": "Only timing is generated, using the slower and more accurate super timing. Use this for variable BPM.",
        "kontextus": "[NONE]",
        "kimenet": "[TIMING]",
        "hozzaad": False,
        "osz": False,
        "super_timing": True,
        "hitsounded": None,
        "referencia": False,
        "szakasz": False,
    },
    "vendeg": {
        "leiras": "Guest difficulty: follow the reference rhythm and timing without copying its patterns. Needs a .osu from the same mapset.",
        "kontextus": "[GD,TIMING,KIAI]",
        "kimenet": "[MAP]",
        "hozzaad": False,
        "osz": False,
        "super_timing": False,
        "hitsounded": None,
        "referencia": True,
        "szakasz": False,
    },
    "hitsound": {
        "leiras": "The reference hit objects stay, and the model adds hitsounds. The .osu is required.",
        "kontextus": "[NO_HS,TIMING,KIAI]",
        "kimenet": "[MAP]",
        "hozzaad": False,
        "osz": False,
        "super_timing": False,
        "hitsounded": True,
        "referencia": True,
        "szakasz": False,
    },
    "szakasz": {
        "leiras": "Regenerate a time range. Hit objects before the start time stay. Set start and end in milliseconds, and provide the .osu.",
        "kontextus": "[TIMING,KIAI]",
        "kimenet": "[MAP]",
        "hozzaad": True,
        "osz": False,
        "super_timing": False,
        "hitsounded": None,
        "referencia": True,
        "szakasz": True,
    },
    "egyeni": {
        "leiras": "Every control is editable. -1 means unknown, or take the value from the reference beatmap.",
        "referencia": False,
        "szakasz": True,
    },
}

KONTEXTUSOK = [
    ("No extra context", "[NONE]"),
    ("Existing timing", "[TIMING]"),
    ("Timing and kiai", "[TIMING,KIAI]"),
    ("Reference hit objects", "[TIMING,KIAI,MAP]"),
    ("Other difficulty rhythm", "[GD,TIMING,KIAI]"),
    ("Objects without hitsounds", "[NO_HS,TIMING,KIAI]"),
]
KIMENETEK = [
    ("Timing, objects, and SV", "[TIMING,MAP,SV]"),
    ("Hit objects only", "[MAP]"),
    ("Timing only", "[TIMING]"),
]


def cimke_nev(szoveg):
    szoveg = (szoveg or "").strip()
    if not szoveg:
        return ""
    if " — " in szoveg:
        szoveg = szoveg.split(" — ", 1)[1].strip()
    if szoveg in ISMERT:
        return szoveg
    if f"{szoveg} " in ISMERT:
        return f"{szoveg} "
    return szoveg


def cimke_opciok(gamemode, mind):
    lathato = []
    mar = set()
    for nev, tag in ISMERT.items():
        if nev in mar:
            continue
        szabaly = tag.get("ruleset_id")
        if not mind and szabaly not in (None, gamemode):
            continue
        mar.add(nev)
        lathato.append(nev)
    lathato.sort(key=str.casefold)
    return ["", *lathato]


def szam_vagy_ures(ertek, egesz=False):
    if ertek is None or ertek == -1:
        return None
    return int(ertek) if egesz else float(ertek)


def szoveg_vagy_ures(ertek):
    ertek = (ertek or "").strip()
    return ertek or None


def kontextus_listaba(szoveg):
    belso = szoveg.strip()[1:-1]
    if not belso or belso.upper() == "NONE":
        return [ContextType.NONE]
    return [ContextType[resz.strip().upper()] for resz in belso.split(",") if resz.strip()]


def cimkek_listaja(dobozok):
    eredmeny = []
    for doboz in dobozok:
        nev = cimke_nev(doboz.value)
        if not nev:
            continue
        if nev not in ISMERT:
            raise ValueError(f"Unknown descriptor: {doboz.value}")
        if nev not in eredmeny:
            eredmeny.append(nev)
    return eredmeny or None


def talloz(cel, cim, fajlok=None, mappa=False):
    import tkinter as tk
    from tkinter import filedialog

    gyoker = tk.Tk()
    gyoker.withdraw()
    gyoker.attributes("-topmost", True)
    kezdo = None
    if cel.value and Path(cel.value).exists():
        ut = Path(cel.value)
        kezdo = str(ut if ut.is_dir() else ut.parent)
    elif OSU_SONGS.exists():
        kezdo = str(OSU_SONGS)
    argumentumok = {"title": cim}
    if kezdo:
        argumentumok["initialdir"] = kezdo
    if mappa:
        valasztott = filedialog.askdirectory(**argumentumok)
    else:
        argumentumok["filetypes"] = fajlok
        valasztott = filedialog.askopenfilename(**argumentumok)
    gyoker.destroy()
    if valasztott:
        cel.value = valasztott


def mezo(cim, vezerlo, sugo=""):
    sorok = [widgets.HTML(f"<b>{cim}</b>")]
    if sugo:
        sorok.append(widgets.HTML(f"<span style='color:#555'>{sugo}</span>"))
    sorok.append(vezerlo)
    return widgets.VBox(sorok, layout=widgets.Layout(width="360px", margin="0 16px 12px 0"))


def sor(*mezok):
    return widgets.Box(list(mezok), layout=widgets.Layout(display="flex", flex_flow="row wrap"))


def konfig_betoltese(nev):
    if GlobalHydra.instance().is_initialized():
        GlobalHydra.instance().clear()
    config_dir = (REPO / "configs" / "inference").resolve().as_posix()
    with initialize_config_dir(version_base="1.1", config_dir=config_dir):
        return compose(config_name=nev)


elfogadom = widgets.Checkbox(value=False, description="I accept", indent=False)
feladat = widgets.Dropdown(
    options=[
        ("New beatmap", "uj"),
        ("Timing only", "timing"),
        ("Guest difficulty", "vendeg"),
        ("Hitsounds for a map", "hitsound"),
        ("Remap a section", "szakasz"),
        ("Custom", "egyeni"),
    ],
    value="uj",
    layout=widgets.Layout(width="460px"),
)
feladat_szoveg = widgets.HTML()

hang = widgets.Text(placeholder=r"C:\...\dal.mp3", layout=widgets.Layout(width="520px"))
beatmap = widgets.Text(placeholder=r"C:\...\difficulty.osu  (optional)", layout=widgets.Layout(width="520px"))
kimenet = widgets.Text(value=str(KIMENET), layout=widgets.Layout(width="520px"))
hatter = widgets.Text(placeholder="background, optional", layout=widgets.Layout(width="520px"))

modell = widgets.Dropdown(
    options=[
        ("V32 – recommended, all 4 modes", "v32"),
        ("V32-mini – smaller and faster", "v32-mini"),
        ("V31", "v31"),
        ("V30 – osu!standard only", "v30"),
        ("V29", "v29"),
    ],
    value="v32",
    layout=widgets.Layout(width="460px"),
)
jatekmod = widgets.Dropdown(
    options=[("osu!standard", 0), ("osu!taiko", 1), ("osu!catch", 2), ("osu!mania", 3)],
    value=0,
    layout=widgets.Layout(width="460px"),
)
nehezseg = widgets.FloatText(value=5.0, layout=widgets.Layout(width="140px"))
mapper_id = widgets.IntText(value=-1, layout=widgets.Layout(width="140px"))
ev = widgets.IntText(value=2023, layout=widgets.Layout(width="140px"))
lora = widgets.Text(placeholder="Hugging Face repo or local folder, optional", layout=widgets.Layout(width="520px"))
minden_cimke = widgets.Checkbox(value=False, description="Tags for every mode", indent=False)

pozitiv = [widgets.Combobox(placeholder="search or pick", ensure_option=False, layout=widgets.Layout(width="340px")) for _ in range(3)]
negativ = [widgets.Combobox(placeholder="works when cfg > 1", ensure_option=False, layout=widgets.Layout(width="340px")) for _ in range(3)]

hp = widgets.FloatText(value=5, layout=widgets.Layout(width="120px"))
cs = widgets.FloatText(value=4, layout=widgets.Layout(width="120px"))
od = widgets.FloatText(value=9, layout=widgets.Layout(width="120px"))
ar = widgets.FloatText(value=8, layout=widgets.Layout(width="120px"))
sv = widgets.FloatText(value=1.4, layout=widgets.Layout(width="120px"))
slider_tick = widgets.FloatText(value=1, layout=widgets.Layout(width="120px"))
billentyu = widgets.IntSlider(value=4, min=1, max=18, step=1, layout=widgets.Layout(width="340px"))
hold = widgets.FloatText(value=-1, layout=widgets.Layout(width="120px"))
scroll = widgets.FloatText(value=-1, layout=widgets.Layout(width="120px"))
hitsounded = widgets.Checkbox(value=True, description="Hitsound (mostly mania)", indent=False)

kontextus = widgets.Dropdown(options=KONTEXTUSOK, value="[NONE]", layout=widgets.Layout(width="340px"))
kimenet_tipus = widgets.Dropdown(options=KIMENETEK, value="[TIMING,MAP,SV]", layout=widgets.Layout(width="340px"))
hozzaad = widgets.Checkbox(value=False, description="Add to the reference map", indent=False)
osz = widgets.Checkbox(value=True, description=".osz package (double-click to import)", indent=False)
felulir = widgets.Checkbox(value=False, description="Overwrite the original .osu", indent=False)
kezdet = widgets.IntText(value=-1, layout=widgets.Layout(width="140px"))
veg = widgets.IntText(value=-1, layout=widgets.Layout(width="140px"))
cfg = widgets.FloatSlider(value=1.0, min=1.0, max=5.0, step=0.1, readout_format=".1f", layout=widgets.Layout(width="340px"))
homerseklet = widgets.FloatSlider(value=0.9, min=0.0, max=1.0, step=0.01, readout_format=".2f", layout=widgets.Layout(width="340px"))
seed = widgets.IntText(value=-1, layout=widgets.Layout(width="140px"))
timing_tures = widgets.IntSlider(value=20, min=0, max=100, step=1, layout=widgets.Layout(width="340px"))
super_timing = widgets.Checkbox(value=False, description="Super timing (slower, more accurate)", indent=False)
sugarak = widgets.IntSlider(value=2, min=1, max=4, step=1, layout=widgets.Layout(width="340px"))
iteraciok = widgets.IntSlider(value=20, min=1, max=40, step=1, layout=widgets.Layout(width="340px"))
bpm_kuszob = widgets.FloatSlider(value=0.1, min=0.0, max=1.0, step=0.1, readout_format=".1f", layout=widgets.Layout(width="340px"))

cim = widgets.Text(placeholder="empty: from the reference, or Unknown Title", layout=widgets.Layout(width="340px"))
eloado = widgets.Text(placeholder="empty: from the reference, or Unknown Artist", layout=widgets.Layout(width="340px"))
keszito = widgets.Text(value="Mapperatorinator", layout=widgets.Layout(width="340px"))
verzio = widgets.Text(value="AI", layout=widgets.Layout(width="340px"))
forras = widgets.Text(placeholder="anime, game, album…", layout=widgets.Layout(width="340px"))

gomb = widgets.Button(
    description="Generate",
    button_style="primary",
    icon="play",
    layout=widgets.Layout(width="260px", height="42px"),
)
naplo = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", padding="8px", max_height="420px", overflow_y="auto"))


def frissit_cimkek(*_):
    opciok = cimke_opciok(int(jatekmod.value), mind=minden_cimke.value)
    ervenyes = {cimke_nev(opcio): opcio for opcio in opciok if opcio}
    for doboz in (*pozitiv, *negativ):
        nev = cimke_nev(doboz.value)
        doboz.options = opciok
        doboz.value = ervenyes.get(nev, "")


def zarolas():
    egyeni = feladat.value == "egyeni"
    adat = FELADATOK[feladat.value]
    feladat_szoveg.value = f"<p style='margin:4px 0 8px 0'>{adat['leiras']}</p>"
    for vezerlo in (kontextus, kimenet_tipus, hozzaad, osz, super_timing):
        vezerlo.disabled = not egyeni
    szakasz = egyeni or adat.get("szakasz", False)
    kezdet.disabled = not szakasz
    veg.disabled = not szakasz
    if not egyeni and adat.get("hitsounded") is not None:
        hitsounded.value = adat["hitsounded"]
        hitsounded.disabled = True
    else:
        hitsounded.disabled = False
    felulir.disabled = not (egyeni or hozzaad.value)
    if not hozzaad.value:
        felulir.value = False


def alkalmaz_feladatot(*_):
    if feladat.value == "egyeni":
        zarolas()
        return
    adat = FELADATOK[feladat.value]
    kontextus.value = adat["kontextus"]
    kimenet_tipus.value = adat["kimenet"]
    hozzaad.value = adat["hozzaad"]
    osz.value = adat["osz"]
    super_timing.value = adat["super_timing"]
    felulir.value = False
    zarolas()


def allapot():
    return {
        "verzio": 1,
        "elfogadom": elfogadom.value,
        "feladat": feladat.value,
        "hang": hang.value,
        "beatmap": beatmap.value,
        "kimenet": kimenet.value,
        "hatter": hatter.value,
        "modell": modell.value,
        "jatekmod": jatekmod.value,
        "nehezseg": nehezseg.value,
        "mapper_id": mapper_id.value,
        "ev": ev.value,
        "lora": lora.value,
        "minden_cimke": minden_cimke.value,
        "pozitiv": [doboz.value for doboz in pozitiv],
        "negativ": [doboz.value for doboz in negativ],
        "hp": hp.value,
        "cs": cs.value,
        "od": od.value,
        "ar": ar.value,
        "sv": sv.value,
        "slider_tick": slider_tick.value,
        "billentyu": billentyu.value,
        "hold": hold.value,
        "scroll": scroll.value,
        "hitsounded": hitsounded.value,
        "kontextus": kontextus.value,
        "kimenet_tipus": kimenet_tipus.value,
        "hozzaad": hozzaad.value,
        "osz": osz.value,
        "felulir": felulir.value,
        "kezdet": kezdet.value,
        "veg": veg.value,
        "cfg": cfg.value,
        "homerseklet": homerseklet.value,
        "seed": seed.value,
        "timing_tures": timing_tures.value,
        "super_timing": super_timing.value,
        "sugarak": sugarak.value,
        "iteraciok": iteraciok.value,
        "bpm_kuszob": bpm_kuszob.value,
        "cim": cim.value,
        "eloado": eloado.value,
        "keszito": keszito.value,
        "diff_verzio": verzio.value,
        "forras": forras.value,
    }


def ertek_beallit(vezerlo, ertek):
    if ertek is not None:
        vezerlo.value = ertek


def betolt():
    if not BEALLITASOK.exists():
        frissit_cimkek()
        alkalmaz_feladatot()
        return
    adat = json.loads(BEALLITASOK.read_text(encoding="utf-8"))
    ertek_beallit(elfogadom, adat.get("elfogadom"))
    ertek_beallit(feladat, adat.get("feladat"))
    ertek_beallit(hang, adat.get("hang"))
    ertek_beallit(beatmap, adat.get("beatmap"))
    ertek_beallit(kimenet, adat.get("kimenet"))
    ertek_beallit(hatter, adat.get("hatter"))
    ertek_beallit(modell, adat.get("modell"))
    ertek_beallit(jatekmod, adat.get("jatekmod"))
    ertek_beallit(nehezseg, adat.get("nehezseg"))
    ertek_beallit(mapper_id, adat.get("mapper_id"))
    ertek_beallit(ev, adat.get("ev"))
    ertek_beallit(lora, adat.get("lora"))
    ertek_beallit(minden_cimke, adat.get("minden_cimke"))
    frissit_cimkek()
    for doboz, mentett in zip(pozitiv, adat.get("pozitiv", [])):
        doboz.value = mentett
    for doboz, mentett in zip(negativ, adat.get("negativ", [])):
        doboz.value = mentett
    for vezerlo, kulcs in (
        (hp, "hp"), (cs, "cs"), (od, "od"), (ar, "ar"), (sv, "sv"), (slider_tick, "slider_tick"),
        (billentyu, "billentyu"), (hold, "hold"), (scroll, "scroll"), (hitsounded, "hitsounded"),
        (kontextus, "kontextus"), (kimenet_tipus, "kimenet_tipus"), (hozzaad, "hozzaad"),
        (osz, "osz"), (felulir, "felulir"), (kezdet, "kezdet"), (veg, "veg"), (cfg, "cfg"),
        (homerseklet, "homerseklet"), (seed, "seed"), (timing_tures, "timing_tures"),
        (super_timing, "super_timing"), (sugarak, "sugarak"), (iteraciok, "iteraciok"),
        (bpm_kuszob, "bpm_kuszob"), (cim, "cim"), (eloado, "eloado"), (keszito, "keszito"),
        (verzio, "diff_verzio"), (forras, "forras"),
    ):
        ertek_beallit(vezerlo, adat.get(kulcs))
    zarolas()


def mentes():
    BEALLITASOK.write_text(json.dumps(allapot(), ensure_ascii=False, indent=2), encoding="utf-8")


class Naplo:
    encoding = "utf-8"

    def __init__(self, eredeti):
        self.eredeti = eredeti
        self._darab = ""
        self._utolso = 0.0

    def write(self, szoveg):
        if not szoveg:
            return
        self.eredeti.write(szoveg)
        self._darab += szoveg
        if "\r" not in self._darab and "\n" not in self._darab:
            return
        sorok = self._darab.replace("\r", "\n").split("\n")
        self._darab = sorok[-1]
        most = time.monotonic()
        for sor in sorok[:-1]:
            sor = sor.strip()
            if not sor:
                continue
            folyamat = "%" in sor or "it/s" in sor or "MB/s" in sor
            if folyamat and most - self._utolso < 1.0:
                continue
            if folyamat:
                self._utolso = most
            try:
                naplo.append_stdout(sor + "\n")
            except Exception:
                pass

    def flush(self):
        if self._darab.strip():
            try:
                naplo.append_stdout(self._darab.strip() + "\n")
            except Exception:
                pass
            self._darab = ""
        self.eredeti.flush()

    def isatty(self):
        return True

    def writable(self):
        return True

    def __getattr__(self, nev):
        return getattr(self.eredeti, nev)


def generalas(_=None):
    gomb.disabled = True
    regi_felirat = gomb.description
    gomb.description = "Generating..."
    naplo.clear_output(wait=True)
    eredeti_konyvtar = os.getcwd()
    eredeti_ki = sys.stdout
    eredeti_be = sys.stderr
    sys.stdout = Naplo(eredeti_ki)
    sys.stderr = Naplo(eredeti_be)
    try:
        mentes()
        if not elfogadom.value:
            raise RuntimeError("Accept the rules at the top before generating.")

        hang_ut = szoveg_vagy_ures(hang.value)
        beatmap_ut = szoveg_vagy_ures(beatmap.value)
        kimenet_ut = szoveg_vagy_ures(kimenet.value) or str(KIMENET)
        if not hang_ut and not beatmap_ut:
            raise RuntimeError("Provide audio, or a .osu with the audio next to it.")
        if hang_ut and not Path(hang_ut).is_file():
            raise RuntimeError(f"Audio file not found: {hang_ut}")
        if beatmap_ut and not Path(beatmap_ut).is_file():
            raise RuntimeError(f"Reference .osu not found: {beatmap_ut}")
        if beatmap_ut and Path(beatmap_ut).suffix.lower() != ".osu":
            raise RuntimeError("The reference must be a .osu file.")
        if hang_ut and Path(hang_ut).suffix.lower() not in {".mp3", ".ogg", ".wav", ".flac", ".m4a"}:
            raise RuntimeError("Audio must be mp3, ogg, wav, flac, or m4a.")
        kezdo_ms = szam_vagy_ures(kezdet.value, egesz=True)
        veg_ms = szam_vagy_ures(veg.value, egesz=True)
        if kezdo_ms is not None and veg_ms is not None and kezdo_ms >= veg_ms:
            raise RuntimeError("The section end must be later than the start.")
        if felulir.value and not beatmap_ut:
            raise RuntimeError("Overwriting needs the reference .osu.")

        kontextus_ertek = kontextus_listaba(kontextus.value)
        kimenet_ertek = kontextus_listaba(kimenet_tipus.value)
        if modell.value in ("v32", "v32-mini") and ContextType.KIAI in kimenet_ertek:
            print("Warning: V32 does not generate a separate kiai track. Output is timing, hit objects, and SV.")
            kimenet_ertek = [elem for elem in kimenet_ertek if elem != ContextType.KIAI] or [ContextType.TIMING, ContextType.MAP, ContextType.SV]
        cimke_ertek = cimkek_listaja(pozitiv)
        negativ_ertek = cimkek_listaja(negativ)
        if modell.value == "v30":
            if int(jatekmod.value) != 0:
                raise RuntimeError("V30 only supports osu!standard. Pick V32, or set the mode to standard.")
            tiltott = {ContextType.KIAI, ContextType.MAP, ContextType.SV}
            if any(elem in tiltott for elem in kontextus_ertek):
                print("Warning: V30 ignores KIAI, MAP, and SV context.")
                kontextus_ertek = [elem for elem in kontextus_ertek if elem not in tiltott] or [ContextType.NONE]
            if kimenet_ertek != [ContextType.MAP]:
                print("Warning: V30 only writes hit objects, so the output is MAP.")
                kimenet_ertek = [ContextType.MAP]
            if cimke_ertek or negativ_ertek:
                print("Warning: V30 ignores descriptors.")
                cimke_ertek = None
                negativ_ertek = None
            if super_timing.value:
                print("Warning: V30 super timing is very slow.")
        if negativ_ertek and cfg.value <= 1:
            print("Warning: negative descriptors only apply when cfg is greater than 1.")
        kell_referencia = bool(hozzaad.value) or any(
            elem in kontextus_ertek
            for elem in (
                ContextType.TIMING,
                ContextType.KIAI,
                ContextType.MAP,
                ContextType.SV,
                ContextType.GD,
                ContextType.NO_HS,
            )
        )
        if kell_referencia and not beatmap_ut:
            raise RuntimeError("This setup needs a reference .osu file.")

        Path(kimenet_ut).mkdir(parents=True, exist_ok=True)
        conf = konfig_betoltese(modell.value)
        conf.audio_path = hang_ut or ""
        conf.output_path = kimenet_ut
        conf.beatmap_path = beatmap_ut or ""
        conf.gamemode = int(jatekmod.value)
        conf.difficulty = szam_vagy_ures(nehezseg.value)
        conf.mapper_id = szam_vagy_ures(mapper_id.value, egesz=True)
        conf.year = szam_vagy_ures(ev.value, egesz=True)
        conf.lora_path = szoveg_vagy_ures(lora.value)
        conf.hitsounded = bool(hitsounded.value)
        conf.hp_drain_rate = szam_vagy_ures(hp.value)
        conf.circle_size = szam_vagy_ures(cs.value)
        conf.overall_difficulty = szam_vagy_ures(od.value)
        conf.approach_rate = szam_vagy_ures(ar.value)
        conf.slider_multiplier = szam_vagy_ures(sv.value)
        conf.slider_tick_rate = szam_vagy_ures(slider_tick.value)
        conf.keycount = int(billentyu.value)
        conf.hold_note_ratio = szam_vagy_ures(hold.value)
        conf.scroll_speed_ratio = szam_vagy_ures(scroll.value)
        conf.descriptors = cimke_ertek
        conf.negative_descriptors = negativ_ertek
        conf.export_osz = bool(osz.value)
        conf.add_to_beatmap = bool(hozzaad.value)
        conf.overwrite_reference_beatmap = bool(felulir.value)
        conf.start_time = kezdo_ms
        conf.end_time = veg_ms
        conf.in_context = kontextus_ertek
        conf.output_type = kimenet_ertek
        conf.cfg_scale = float(cfg.value)
        conf.temperature = float(homerseklet.value)
        conf.seed = szam_vagy_ures(seed.value, egesz=True)
        conf.timing_leniency = int(timing_tures.value)
        conf.super_timing = bool(super_timing.value)
        conf.timer_num_beams = int(sugarak.value)
        conf.timer_iterations = int(iteraciok.value)
        conf.timer_bpm_threshold = float(bpm_kuszob.value)
        conf.use_server = False
        conf.title = szoveg_vagy_ures(cim.value)
        conf.artist = szoveg_vagy_ures(eloado.value)
        conf.creator = szoveg_vagy_ures(keszito.value)
        conf.version = szoveg_vagy_ures(verzio.value)
        conf.source = szoveg_vagy_ures(forras.value)
        conf.background = szoveg_vagy_ures(hatter.value)

        print("Generation started. The first run downloads several GB; after that a song usually takes a few minutes.")
        if felulir.value:
            print("The original .osu will be overwritten.")
        os.chdir(REPO)
        eredmeny = inference.main(conf)
        if not isinstance(eredmeny, tuple) or len(eredmeny) != 2:
            raise RuntimeError(f"Generation did not return a file: {eredmeny!r}")
        utvonal = Path(eredmeny[1])
        print(f"Kész: {utvonal}")
        if utvonal.suffix.lower() == ".osz":
            print("Double-click the .osz to import it into osu!.")
        else:
            print("Copy the .osu into the song folder, next to the audio.")
        print("Disclose in the beatmap that AI was used.")
        try:
            os.startfile(utvonal.parent)
        except OSError as hiba:
            print(f"Could not open the folder: {hiba}")
    except Exception as hiba:
        print(f"Error: {hiba}")
        if not isinstance(hiba, RuntimeError):
            traceback.print_exc()
    finally:
        sys.stdout = eredeti_ki
        sys.stderr = eredeti_be
        os.chdir(eredeti_konyvtar)
        gomb.disabled = False
        gomb.description = regi_felirat


gomb.on_click(generalas)
hang_gomb = widgets.Button(description="Browse", icon="folder")
beatmap_gomb = widgets.Button(description="Tallózás", icon="folder")
kimenet_gomb = widgets.Button(description="Tallózás", icon="folder")
hatter_gomb = widgets.Button(description="Tallózás", icon="folder")
hang_gomb.on_click(lambda _: talloz(hang, "Hangfájl", [("Hang", "*.mp3 *.ogg *.wav *.flac *.m4a"), ("Minden fájl", "*.*")]))
beatmap_gomb.on_click(lambda _: talloz(beatmap, "Referencia beatmap", [("osu! beatmap", "*.osu"), ("Minden fájl", "*.*")]))
kimenet_gomb.on_click(lambda _: talloz(kimenet, "Kimeneti mappa", mappa=True))
hatter_gomb.on_click(lambda _: talloz(hatter, "Háttérkép", [("Kép", "*.jpg *.jpeg *.png"), ("Minden fájl", "*.*")]))

betolt()
feladat.observe(alkalmaz_feladatot, names="value")
jatekmod.observe(frissit_cimkek, names="value")
minden_cimke.observe(frissit_cimkek, names="value")
hozzaad.observe(lambda _: zarolas(), names="value")

urlap = widgets.VBox([
    widgets.HTML("<h2 style='margin:0 0 8px 0'>Generate a beatmap</h2>"),
    widgets.HTML("<p style='margin:0 0 8px 0'>Always disclose the use of AI in your beatmaps. Generation starts only after you tick the box.</p>"),
    elfogadom,
    mezo("Task", feladat),
    feladat_szoveg,
    widgets.HTML("<h3>Files</h3>"),
    mezo("Audio", widgets.HBox([hang, hang_gomb]), "Leave empty when the audio sits next to the .osu."),
    mezo("Reference beatmap", widgets.HBox([beatmap, beatmap_gomb]), "Required for in-context options and for remapping a section."),
    mezo("Output folder", widgets.HBox([kimenet, kimenet_gomb])),
    widgets.HTML("<h3>Style</h3>"),
    sor(
        mezo("Model", modell, "V32 is the best model. V30 does not support other gamemodes, year, descriptors, or in-context."),
        mezo("Game mode", jatekmod),
    ),
    sor(
        mezo("Star rating", nehezseg, "-1 leaves it unknown. The result can deviate with song intensity and the other settings."),
        mezo("Mapper ID", mapper_id, "The number in the mapper profile URL. -1 means none."),
        mezo("Year", ev, "Prefer a year from 2007 to 2024. -1 leaves it unknown."),
    ),
    mezo("LoRA", lora, "Optional Hugging Face repo or local folder."),
    minden_cimke,
    widgets.HTML("<b>Descriptors</b> <span style='color:#555'>Leave empty to keep the reference tags, or let the model decide on a new map. <a href='https://osu.ppy.sh/wiki/en/Beatmap/Beatmap_tags'>osu! wiki</a></span>"),
    sor(*[mezo(f"{i}. descriptor", doboz) for i, doboz in enumerate(pozitiv, start=1)]),
    widgets.HTML("<b>Negative descriptors</b> <span style='color:#555'>What the beatmap should not have. Only works when cfg &gt; 1, and use the same count as the positive descriptors.</span>"),
    sor(*[mezo(f"{i}. negative", doboz) for i, doboz in enumerate(negativ, start=1)]),
    widgets.Accordion(children=[
        widgets.VBox([
            sor(
                mezo("HP", hp, "automatic"),
                mezo("Circle size", cs, "Mania key count is set separately."),
                mezo("Overall difficulty", od),
                mezo("Approach rate", ar),
                mezo("Slider multiplier", sv, "Usually 0.4–3.6."),
                mezo("Slider tick", slider_tick),
            ),
            sor(
                mezo("Key count", billentyu, "Mania only."),
                mezo("Hold note ratio", hold, "0–1, mania only. -1 is automatic."),
                mezo("Scroll speed ratio", scroll, "0–1, mania and taiko. -1 is automatic."),
            ),
            hitsounded,
        ]),
        widgets.VBox([
            sor(mezo("In context", kontextus, "TIMING skips timing generation. KIAI gives kiai times. MAP gives hit objects. GD uses another difficulty for rhythm. NO_HS copies objects and only adds hitsounds."), mezo("Output", kimenet_tipus, "Generate everything, or timing only.")),
            hozzaad,
            osz,
            felulir,
            sor(
                mezo("Start", kezdet, "-1 is the start of the song."),
                mezo("End", veg, "-1 is the end of the song."),
            ),
            sor(
                mezo("Cfg", cfg, "Higher values follow descriptors and mapper style more strongly. A high cfg can look unexpected."),
                mezo("Temperature", homerseklet, "Lower is more conservative. Only lower it slightly, or when remapping a short section."),
                mezo("Seed", seed, "-1 is random. The same seed samples the same beatmap again."),
            ),
            mezo("Timing leniency", timing_tures, "A higher value keeps fewer timing points."),
            super_timing,
            sor(
                mezo("Super timing beams", sugarak, "More beams are slightly more accurate and slower."),
                mezo("Super timing iterations", iteraciok, "More iterations are slightly more accurate and slower."),
                mezo("BPM-change threshold", bpm_kuszob, "A higher value allows fewer BPM changes."),
            ),
        ]),
        widgets.VBox([
            sor(
                mezo("Title", cim),
                mezo("Artist", eloado),
                mezo("Creator", keszito),
                mezo("Version", verzio),
                mezo("Source", forras),
            ),
            mezo("Background", widgets.HBox([hatter, hatter_gomb])),
        ]),
    ], titles=("Difficulty", "Generation", "Metadata"), selected_index=None),
    widgets.HTML("<div style='height:8px'></div>"),
    gomb,
    naplo,
])
display(urlap)
